# Sequence Models — RNN, LSTM, GRU, and Bidirectional LSTM

**Author:** Shivani Bokka  
**Dataset:** 20 Newsgroups (binary: rec.sport.hockey vs sci.med)  
**Goal:** Understand and compare sequence models for text classification

---

## What Is This Notebook About?

This notebook is a **complete guide to sequence models** — the family of neural networks designed to process data where **order matters**. We start from the basic Recurrent Neural Network (RNN) and work up through LSTMs, GRUs, Bidirectional LSTMs, and attention-augmented models.

Each section explains the core idea in plain English before showing you the code.

---

## Why Do We Need Sequence Models?

Standard neural networks treat each input independently — they have no memory. But many real-world problems involve sequences where context from earlier steps matters:

- **Text:** The word "bank" means something different in "river bank" vs "bank account"
- **Time series:** Today's stock price depends on yesterday's
- **Speech:** Phonemes only make sense in sequence
- **Video:** Frames are not independent images

Sequence models solve this by maintaining a **hidden state** that carries information from one step to the next.

---

## Models Covered

| # | Model | Key Idea |
|---|-------|----------|
| 1 | Vanilla RNN | Simple recurrent unit; suffers from vanishing gradients |
| 2 | LSTM | Long Short-Term Memory; gated cell state for long dependencies |
| 3 | GRU | Gated Recurrent Unit; simpler than LSTM, often just as good |
| 4 | BiLSTM | Bidirectional LSTM; reads sequence forward AND backward |
| 5 | BiLSTM + Attention | Attention over all hidden states; best at capturing key tokens |

---

## Step 1 — Imports and Setup

We import PyTorch for deep learning, sklearn for data loading, and standard utilities for visualization and timing.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('All libraries imported successfully!')

---

## Section 2 — Why Sequence Models?

### The Problem with Order

Consider these two sentences:

> "This movie is **not bad** at all."

> "This movie is **bad**, not good at all."

A bag-of-words model sees the same words in both sentences — "not", "bad", "good" — and might assign similar sentiment scores. But clearly these sentences have **opposite meanings**. The **order** of words is everything.

Sequence models handle this by processing tokens one at a time, updating a hidden state:

```
h_t = f(h_{t-1}, x_t)
```

Where:
- `h_t` = hidden state at step t (the model's memory)
- `h_{t-1}` = previous hidden state
- `x_t` = current input token
- `f` = a learned function

### Applications of Sequence Models

| Domain | Task | Why Order Matters |
|--------|------|-------------------|
| NLP | Sentiment analysis | "not bad" ≠ "bad not" |
| Finance | Stock prediction | Yesterday's price informs today's |
| Medicine | ECG analysis | Heart rhythm is a time sequence |
| Music | Generation | Notes depend on preceding melody |
| Video | Action recognition | Frame 5 depends on frames 1–4 |

---

---

## Section 3 — The Vanishing Gradient Problem in RNNs

### What Is a Vanishing Gradient?

During backpropagation through time (BPTT), gradients are multiplied at each time step. If the weight matrix has values less than 1, repeated multiplication drives the gradient toward zero — the **vanishing gradient problem**.

Mathematically, the gradient at step t=0 involves a product:

```
∂L/∂h_0 = (∂h_T/∂h_{T-1}) × (∂h_{T-1}/∂h_{T-2}) × ... × (∂h_1/∂h_0)
```

Each term ∂h_t/∂h_{t-1} = W_h × diag(tanh'(h_{t-1})), and since tanh' ≤ 1, if |W_h| < 1, the product shrinks exponentially.

**The result:** The model cannot learn dependencies that span more than ~5-10 steps. It literally "forgets" what happened early in the sequence.

Let's demonstrate this with a manual numpy simulation.

In [ ]:
# Simulate vanishing gradients in a vanilla RNN over 10 timesteps
np.random.seed(42)

T = 10          # Number of timesteps
hidden_size = 4

# Initialize a weight matrix with spectral radius slightly < 1
W = np.random.randn(hidden_size, hidden_size) * 0.5

def tanh_deriv(x):
    """Derivative of tanh: 1 - tanh(x)^2"""
    return 1.0 - np.tanh(x)**2

# Simulate a forward pass storing hidden states
h = np.zeros((T+1, hidden_size))
x_seq = np.random.randn(T, hidden_size)

for t in range(T):
    h[t+1] = np.tanh(W @ h[t] + x_seq[t])

# Compute gradient norms step by step (backprop through time)
# Starting from a unit gradient at the last step
grad = np.ones(hidden_size)  # dL/dh_T
grad_norms = [np.linalg.norm(grad)]

for t in range(T-1, -1, -1):
    # Jacobian of h_{t+1} w.r.t. h_t: W^T diag(tanh'(h_{t+1}))
    diag = tanh_deriv(W @ h[t] + x_seq[t])
    jacobian = W.T * diag  # element-wise broadcast
    grad = jacobian @ grad
    grad_norms.append(np.linalg.norm(grad))

# grad_norms[0] = at step T, grad_norms[-1] = at step 0
steps = list(range(T+1))
# Reverse so index 0 = earliest timestep
grad_norms_plot = grad_norms[::-1]

print('Gradient norms from t=0 to t=T:')
for i, g in enumerate(grad_norms_plot):
    print(f'  Step {i:2d}: gradient norm = {g:.6f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(steps, grad_norms_plot, color='steelblue', edgecolor='white')
ax.set_xlabel('Timestep (0 = earliest in sequence)', fontsize=12)
ax.set_ylabel('Gradient Norm', fontsize=12)
ax.set_title('Vanishing Gradient in a Vanilla RNN over 10 Timesteps', fontsize=13)
ax.set_xticks(steps)
plt.tight_layout()
plt.show()

### How to Read This Chart: Vanishing Gradient in RNNs

This bar chart shows the **gradient norm** at each timestep of a 10-step RNN, measured during backpropagation from the final step backward to step 0.

- **The x-axis** shows the timestep, with 0 being the **earliest** token in the sequence and 10 being the **latest**.
- **The y-axis** shows the magnitude (norm) of the gradient signal at that timestep.
- **Tall bars** = the model is still receiving a strong learning signal — it can update its weights based on that step.
- **Short bars** = the gradient has nearly vanished — the model learns almost nothing about that early timestep.

**What to look for:**
- Notice how the gradient norm is **large at step 10** (most recent) and **nearly zero at step 0** (earliest).
- This means a vanilla RNN trained on 10-step sequences will barely learn from the first few tokens.
- For longer sequences (50, 100 steps), the problem is catastrophically worse.

> **This is exactly why LSTMs were invented** — they solve the vanishing gradient problem using gated cell states.

---

## Section 4 — LSTM: Long Short-Term Memory

### What Is an LSTM?

An LSTM is a special kind of RNN that uses **gates** to control what information is remembered or forgotten. It maintains two kinds of state:
- **h_t** — the hidden state (short-term memory, used as output)
- **C_t** — the cell state (long-term memory, flows with minimal modification)

### The Four Gates

```
INPUT:  x_t (current token) and h_{t-1} (previous hidden state)

┌─────────────────────────────────────────────────────┐
│  FORGET GATE: f_t = σ(W_f·[h_{t-1}, x_t] + b_f)   │  ← "How much of C_{t-1} to keep?"
│  INPUT GATE:  i_t = σ(W_i·[h_{t-1}, x_t] + b_i)   │  ← "How much new info to add?"
│  CANDIDATE:   g_t = tanh(W_g·[h_{t-1}, x_t] + b_g)│  ← "What new info to add?"
│  OUTPUT GATE: o_t = σ(W_o·[h_{t-1}, x_t] + b_o)   │  ← "What to output from cell?"
│                                                     │
│  CELL UPDATE: C_t = f_t ⊙ C_{t-1} + i_t ⊙ g_t     │  ← New long-term memory
│  HIDDEN:      h_t = o_t ⊙ tanh(C_t)                │  ← New short-term memory
└─────────────────────────────────────────────────────┘
```

The key insight: the cell state C_t is updated by **addition**, not multiplication. This means gradients can flow backward through many steps without vanishing.

### Plain English Explanation

- **Forget gate**: "Reading a new article about Topic B — forget what I knew about Topic A."
- **Input gate**: "This sentence is very important — write it to long-term memory."
- **Output gate**: "Given my long-term memory, what's relevant to say right now?"

Let's train an LSTM on a sine wave prediction task.

In [ ]:
# Train LSTM on sine wave prediction
torch.manual_seed(42)

# Generate sine wave data
t = np.linspace(0, 4 * np.pi, 400)
sine = np.sin(t).astype(np.float32)

SEQ_LEN = 50

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X_sine, y_sine = make_sequences(sine, SEQ_LEN)
X_sine = torch.tensor(X_sine).unsqueeze(-1)  # (N, 50, 1)
y_sine = torch.tensor(y_sine).unsqueeze(-1)  # (N, 1)

class SineLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # Use last hidden state

class SineGRU(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

def train_sine_model(model, X, y, epochs=80, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    losses = []
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

lstm_model = SineLSTM()
lstm_losses = train_sine_model(lstm_model, X_sine, y_sine)

# Plot predictions vs actual
lstm_model.eval()
with torch.no_grad():
    preds = lstm_model(X_sine).squeeze().numpy()

actual = y_sine.squeeze().numpy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(actual, label='Actual Sine Wave', color='steelblue', alpha=0.7)
ax.plot(preds, label='LSTM Predictions', color='tomato', linestyle='--', alpha=0.9)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('LSTM Sine Wave Prediction: Actual vs Predicted')
ax.legend()
plt.tight_layout()
plt.show()

### How to Read This Chart: LSTM Sine Wave Prediction

This line chart compares the LSTM's predictions to the actual sine wave values.

- **The blue line** is the ground truth — the actual sine wave values computed with `np.sin(t)`.
- **The red dashed line** is what the LSTM predicted, given the previous 50 steps as input.
- **The x-axis** is the timestep index within the sine wave.
- **The y-axis** is the value (ranging from -1 to +1 for a standard sine wave).

**What to look for:**
- If the red line closely follows the blue line, the LSTM has learned the periodic pattern of the sine wave.
- Any lag (the red line slightly behind the blue) is normal — the model is predicting one step ahead.
- A smooth, accurate prediction shows the LSTM can capture long-range temporal dependencies (remembering the phase of the wave across 50 steps).

> **Key takeaway:** Even simple LSTMs can model periodic signals with high accuracy — something a vanilla RNN struggles with on longer sequences.

---

## Section 5 — GRU vs LSTM

### What Is a GRU?

The **Gated Recurrent Unit (GRU)** was introduced in 2014 as a simplification of the LSTM. It achieves similar performance with fewer parameters by merging the forget and input gates into a single **update gate**, and eliminating the separate cell state.

```
GRU Gates:
  Reset gate:  r_t = σ(W_r·[h_{t-1}, x_t])
  Update gate: z_t = σ(W_z·[h_{t-1}, x_t])
  Candidate:   n_t = tanh(W_n·[r_t ⊙ h_{t-1}, x_t])
  New hidden:  h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ n_t
```

### LSTM vs GRU: When to Choose Which?

| Property | LSTM | GRU |
|----------|------|-----|
| Gates | 3 (forget, input, output) | 2 (reset, update) |
| Parameters | More | ~25% fewer |
| Training speed | Slower | Faster |
| Performance on long sequences | Slightly better | Close to LSTM |
| Best for | Complex long-term patterns | When compute is limited |

**Practical advice:** Try GRU first. If it's not performing well enough, switch to LSTM.

In [ ]:
# Train GRU on same sine wave and compare loss curves
torch.manual_seed(42)

gru_model = SineGRU()
gru_losses = train_sine_model(gru_model, X_sine, y_sine)

# Retrain LSTM with same seed for fair comparison
torch.manual_seed(42)
lstm_model2 = SineLSTM()
lstm_losses2 = train_sine_model(lstm_model2, X_sine, y_sine)

epochs_range = list(range(1, 81))
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, lstm_losses2, label='LSTM Loss', color='steelblue')
ax.plot(epochs_range, gru_losses, label='GRU Loss', color='tomato', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('GRU vs LSTM Training Curves on Sine Wave Prediction')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Final LSTM Loss: {lstm_losses2[-1]:.6f}')
print(f'Final GRU Loss:  {gru_losses[-1]:.6f}')

### How to Read This Chart: GRU vs LSTM Training Curves

This line chart overlays the training loss curves of GRU and LSTM on the same sine wave prediction task.

- **The x-axis** is the training epoch (one full pass through the dataset).
- **The y-axis** is Mean Squared Error (MSE) loss — lower is better.
- **Blue line** = LSTM loss per epoch.
- **Red dashed line** = GRU loss per epoch.

**What to look for:**
- Both models should converge to similar final loss values — GRU and LSTM are roughly equivalent on simple tasks.
- GRU may converge slightly faster (fewer parameters = less computation per step).
- Large differences between the curves would suggest one architecture is significantly better suited for this task.

> **Practical note:** On many NLP benchmark tasks, GRU and LSTM perform within 1-2% of each other. The choice often comes down to available compute and training time.

---

## Section 6 — Loading and Preprocessing the 20 Newsgroups Dataset

### The Dataset

We use two categories from the 20 Newsgroups dataset as a binary text classification proxy:
- **rec.sport.hockey** → Label 0
- **sci.med** → Label 1

### Preprocessing Pipeline

1. Load raw text documents
2. Build a vocabulary of the top 5000 most frequent words
3. Convert each document to a sequence of word indices
4. Pad short sequences with 0s; truncate long sequences to length 100
5. Create PyTorch DataLoaders for training and validation

In [ ]:
# Load 20 newsgroups data
categories = ['rec.sport.hockey', 'sci.med']
newsgroups = fetch_20newsgroups(subset='all', categories=categories,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=42)

texts = newsgroups.data
labels = newsgroups.target  # 0 = hockey, 1 = sci.med

print(f'Total documents: {len(texts)}')
print(f'Label distribution: hockey={sum(labels==0)}, sci.med={sum(labels==1)}')
print(f'Sample text (first 200 chars):\n{texts[0][:200]}')

# ---- Build vocabulary ----
VOCAB_SIZE = 5000
MAX_LEN = 100
PAD_IDX = 0

def simple_tokenize(text):
    """Lowercase and split on whitespace/punctuation."""
    import re
    return re.findall(r'[a-z]+', text.lower())

# Build word frequency counter
counter = Counter()
for text in texts:
    counter.update(simple_tokenize(text))

# Most common VOCAB_SIZE words; index 0 = PAD, 1 = UNK
most_common = [word for word, _ in counter.most_common(VOCAB_SIZE - 2)]
word2idx = {word: idx+2 for idx, word in enumerate(most_common)}
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1

def encode(text, word2idx, max_len):
    tokens = simple_tokenize(text)
    indices = [word2idx.get(t, 1) for t in tokens]  # 1 = UNK
    # Truncate or pad
    if len(indices) >= max_len:
        return indices[:max_len]
    else:
        return indices + [0] * (max_len - len(indices))

# Encode all documents
X_encoded = np.array([encode(t, word2idx, MAX_LEN) for t in texts], dtype=np.int64)
y_labels = np.array(labels, dtype=np.int64)

print(f'\nEncoded shape: {X_encoded.shape}')
print(f'Vocab size: {len(word2idx)}')
print(f'Sample encoded (first 20 tokens): {X_encoded[0][:20]}')

In [ ]:
# Create PyTorch Dataset and DataLoaders
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Train/val split
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_encoded, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

train_dataset = TextDataset(X_train_raw, y_train_raw)
val_dataset = TextDataset(X_val_raw, y_val_raw)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Training batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')

---

## Section 7 — pack_padded_sequence: Processing Variable-Length Sequences Efficiently

### The Problem with Padding

When we pad sequences to a fixed length, we're making the RNN process many zero-padded tokens that contribute nothing. This wastes computation and can confuse the model (it updates hidden states on pad tokens).

### The Solution: PackedSequence

PyTorch's `pack_padded_sequence` tells the RNN:
> "Each sequence in this batch has a different real length. Skip the padding."

The packed representation stores only the real tokens, sorted by sequence length. After the RNN, `pad_packed_sequence` converts back to a padded tensor.

```
WITHOUT PACKING:
  Batch: [[1,2,3,0,0], [4,5,0,0,0], [6,7,8,9,10]]  ← RNN processes all 15 tokens

WITH PACKING:
  Sorted: [[6,7,8,9,10], [1,2,3,0,0], [4,5,0,0,0]]
  Packed: [6,1,4, 7,2,5, 8,3, 9, 10]              ← RNN processes only 10 real tokens
```

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Demo: forward pass with and without packing
torch.manual_seed(0)

# Create a small batch of 3 sequences with different real lengths
# (All padded to length 5 with zeros)
batch = torch.tensor([
    [10, 20, 30, 40, 50],  # real length = 5
    [11, 22, 33,  0,  0],  # real length = 3
    [15, 25,  0,  0,  0],  # real length = 2
], dtype=torch.long)

lengths = torch.tensor([5, 3, 2])  # actual (non-padded) lengths

embed = nn.Embedding(100, 8)  # tiny embedding for demo
rnn = nn.RNN(8, 16, batch_first=True)

x_embedded = embed(batch)  # (3, 5, 8)

# --- WITHOUT packing ---
out_no_pack, h_no_pack = rnn(x_embedded)
print('WITHOUT packing:')
print(f'  Output shape: {out_no_pack.shape}')  # (3, 5, 16)
print(f'  Hidden state shape: {h_no_pack.shape}')  # (1, 3, 16)
print(f'  Note: hidden state includes updates from PAD tokens!')

# --- WITH packing ---
# Must sort by descending length first
lengths_sorted, sort_idx = lengths.sort(descending=True)
x_sorted = x_embedded[sort_idx]

packed = pack_padded_sequence(x_sorted, lengths_sorted, batch_first=True)
out_packed, h_packed = rnn(packed)
out_unpacked, lens_unpacked = pad_packed_sequence(out_packed, batch_first=True)

print('\nWITH packing:')
print(f'  Unpacked output shape: {out_unpacked.shape}')  # (3, 5, 16)
print(f'  Hidden state shape: {h_packed.shape}')  # (1, 3, 16)
print(f'  Actual lengths after unpack: {lens_unpacked.tolist()}')
print(f'  Note: hidden state reflects ONLY real tokens — no PAD contamination!')

---

## Section 8 — Gradient Clipping: Preventing Exploding Gradients

### The Exploding Gradient Problem

The opposite of vanishing gradients: if the RNN weight matrix has values greater than 1, repeated multiplication causes gradients to **explode** — growing exponentially during backpropagation. This causes wild parameter updates and NaN losses.

### The Fix: Gradient Clipping

Before the optimizer step, we compute the **global norm** of all gradients and rescale them if they exceed a threshold:

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

If `global_norm > max_norm`, each gradient is scaled by `max_norm / global_norm`. This preserves the **direction** of the gradient but limits its **magnitude**.

Let's train a vanilla RNN on 20newsgroups with and without clipping and compare gradient norms.

In [ ]:
# Gradient clipping comparison on vanilla RNN
torch.manual_seed(42)

class VanillaRNNClip(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=64, hidden_size=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        emb = self.embed(x)
        out, h = self.rnn(emb)
        return self.fc(h.squeeze(0))

def train_with_clip(use_clipping, steps=100):
    torch.manual_seed(42)
    model = VanillaRNNClip()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    grad_norms = []
    model.train()
    loader_iter = iter(train_loader)
    for step in range(steps):
        try:
            xb, yb = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            xb, yb = next(loader_iter)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        # Compute gradient norm BEFORE clipping
        total_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        grad_norms.append(total_norm)
        if use_clipping:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    return grad_norms

print('Training RNN WITHOUT clipping...')
norms_no_clip = train_with_clip(use_clipping=False, steps=100)
print('Training RNN WITH clipping (max_norm=1.0)...')
norms_clip = train_with_clip(use_clipping=True, steps=100)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(norms_no_clip, label='Without Clipping', color='tomato', alpha=0.8)
ax.plot(norms_clip, label='With Clipping (max=1.0)', color='steelblue', alpha=0.8)
ax.axhline(y=1.0, color='black', linestyle=':', linewidth=1, label='Clip Threshold')
ax.set_xlabel('Training Step')
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Norms: With vs Without Gradient Clipping')
ax.legend()
plt.tight_layout()
plt.show()

### How to Read This Chart: Gradient Norms With/Without Clipping

This line chart shows the **global gradient norm** at each training step for two versions of the same RNN — one with gradient clipping and one without.

- **The x-axis** is the training step (each step = one mini-batch).
- **The y-axis** is the global gradient norm (L2 norm across all model parameters).
- **Red line** = gradient norms WITHOUT clipping — can spike dramatically.
- **Blue line** = gradient norms WITH clipping (max_norm=1.0) — never exceed the dotted threshold line.
- **The black dotted line** at y=1.0 is the clip threshold.

**What to look for:**
- Large spikes in the red line indicate **exploding gradient events** — these can destabilize training and lead to NaN losses.
- The blue line stays flat at or below 1.0 whenever the gradient norm would have exceeded the threshold.
- In practice, clipping does not hurt convergence — it just prevents catastrophic update steps.

> **Rule of thumb:** Always use gradient clipping when training RNNs. `max_norm=1.0` is a good default for most tasks.

---

## Section 9 — Defining and Training All Four Models

We now define VanillaRNN, LSTM, GRU, and BiLSTM, all with identical hyperparameters:
- Embedding dimension: 128
- Hidden size: 128  
- 2 layers
- Dropout: 0.3

We train each for 5 epochs and record validation accuracy and wall-clock training time.

In [ ]:
EMBED_DIM = 128
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3
NUM_CLASSES = 2
EPOCHS = 5

class VanillaRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, batch_first=True,
                          dropout=DROPOUT if NUM_LAYERS > 1 else 0)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_SIZE, NUM_CLASSES)

    def forward(self, x):
        emb = self.drop(self.embed(x))
        _, h = self.rnn(emb)
        return self.fc(self.drop(h[-1]))


class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, batch_first=True,
                            dropout=DROPOUT if NUM_LAYERS > 1 else 0)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_SIZE, NUM_CLASSES)

    def forward(self, x):
        emb = self.drop(self.embed(x))
        _, (h, _) = self.lstm(emb)
        return self.fc(self.drop(h[-1]))


class GRUClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=PAD_IDX)
        self.gru = nn.GRU(EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, batch_first=True,
                          dropout=DROPOUT if NUM_LAYERS > 1 else 0)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_SIZE, NUM_CLASSES)

    def forward(self, x):
        emb = self.drop(self.embed(x))
        _, h = self.gru(emb)
        return self.fc(self.drop(h[-1]))


class BiLSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, batch_first=True,
                            bidirectional=True,
                            dropout=DROPOUT if NUM_LAYERS > 1 else 0)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_SIZE * 2, NUM_CLASSES)  # *2 for bidirectional

    def forward(self, x):
        emb = self.drop(self.embed(x))
        _, (h, _) = self.lstm(emb)
        # Concatenate final forward and backward hidden states
        h_cat = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(h_cat))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, cls in [('VanillaRNN', VanillaRNN), ('LSTM', LSTMClassifier),
                   ('GRU', GRUClassifier), ('BiLSTM', BiLSTMClassifier)]:
    m = cls()
    print(f'{name}: {count_parameters(m):,} trainable parameters')

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, clip=1.0):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    val_accs = []
    start = time.time()
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
        # Validate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        acc = correct / total
        val_accs.append(acc)
        print(f'  Epoch {epoch+1}/{epochs} | Val Acc: {acc:.4f}')
    elapsed = time.time() - start
    return val_accs, elapsed

results = {}
model_configs = [
    ('VanillaRNN', VanillaRNN()),
    ('LSTM', LSTMClassifier()),
    ('GRU', GRUClassifier()),
    ('BiLSTM', BiLSTMClassifier()),
]

for name, model in model_configs:
    print(f'\nTraining {name}...')
    torch.manual_seed(42)
    val_accs, elapsed = train_model(model, train_loader, val_loader)
    results[name] = {'val_accs': val_accs, 'time': elapsed, 'final_acc': val_accs[-1]}
    print(f'  Total time: {elapsed:.1f}s | Final Val Acc: {val_accs[-1]:.4f}')

---

## Section 10 — Comparing Model Performance

Let's visualize validation accuracy across epochs for all four models, and compare training time.

In [ ]:
# Line chart: val accuracy per epoch for all 4 models
epochs_range = list(range(1, EPOCHS + 1))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

fig, ax = plt.subplots(figsize=(10, 5))
for (name, _), color in zip(model_configs, colors):
    ax.plot(epochs_range, results[name]['val_accs'], label=name, color=color, marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Validation Accuracy per Epoch: All 4 Models')
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

### How to Read This Chart: Model Comparison

This line chart shows validation accuracy for four sequence models over 5 training epochs.

- **The x-axis** is the epoch number. Each epoch = one full pass through the training data.
- **The y-axis** is validation accuracy — the fraction of held-out examples classified correctly.
- **Each colored line** represents a different model architecture.
- **Dots on each line** mark the accuracy at the end of each epoch.

**What to look for:**
- **Higher lines** = better performing models.
- **Steeper early rise** = faster convergence — the model learns the task quickly.
- **Plateau near the top** = the model has reached its performance ceiling on this task and dataset.
- BiLSTM typically outperforms the others because it reads the sequence in both directions.
- VanillaRNN typically has the lowest accuracy — it struggles with dependencies longer than a few steps.

> **Important:** Validation accuracy (not training accuracy) is the true measure of model quality. A model that scores 99% on training but 60% on validation is overfitting.

In [ ]:
# Bar chart: training time
names = [n for n, _ in model_configs]
times = [results[n]['time'] for n in names]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, times, color=colors, edgecolor='white')
ax.set_xlabel('Model')
ax.set_ylabel('Training Time (seconds)')
ax.set_title('Training Time Comparison Across Models (5 Epochs)')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{t:.1f}s', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

### How to Read This Chart: Training Time Comparison

This bar chart shows the total wall-clock training time (in seconds) for each model over 5 epochs.

- **Each bar** represents one model architecture.
- **Taller bars** = more time to train.
- **Numbers above each bar** show the exact training time in seconds.

**What to look for:**
- VanillaRNN is typically the **fastest** — it has the simplest recurrent computation (no gates).
- LSTM is **slower than GRU** because it has more gates and parameters.
- BiLSTM is typically the **slowest** — it runs two LSTMs (one forward, one backward) in parallel.
- The time cost of BiLSTM is roughly 2x that of a unidirectional LSTM.

> **Trade-off:** BiLSTM gives the best accuracy but costs the most compute. GRU offers a sweet spot between speed and accuracy for most NLP tasks.

---

## Section 11 — Stacking LSTM Layers

### Why Stack Layers?

Just like deep CNNs learn hierarchical image features (edges → shapes → objects), **stacked RNNs** learn hierarchical temporal features:

- **Layer 1:** Low-level patterns (individual words, short n-grams)
- **Layer 2:** Mid-level patterns (phrases, clauses)
- **Layer 3:** High-level patterns (sentence-level semantics, argument structure)

Each layer takes the **sequence of hidden states** from the previous layer as its input.

### The Catch

More layers = more parameters = more risk of overfitting. For short sequences or small datasets, 1-2 layers is usually enough. Deep LSTMs (3+ layers) are typically used with:
- Large datasets
- Strong regularization (dropout)
- Long sequences where hierarchical patterns matter

Let's train 1-layer, 2-layer, and 3-layer LSTMs on our newsgroups task.

In [ ]:
def make_lstm(num_layers):
    class StackedLSTM(nn.Module):
        def __init__(self, n_layers):
            super().__init__()
            self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=PAD_IDX)
            self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_SIZE, n_layers, batch_first=True,
                                dropout=DROPOUT if n_layers > 1 else 0)
            self.drop = nn.Dropout(DROPOUT)
            self.fc = nn.Linear(HIDDEN_SIZE, NUM_CLASSES)

        def forward(self, x):
            emb = self.drop(self.embed(x))
            _, (h, _) = self.lstm(emb)
            return self.fc(self.drop(h[-1]))

    return StackedLSTM(num_layers)

stack_results = {}
for n_layers in [1, 2, 3]:
    print(f'\nTraining {n_layers}-layer LSTM...')
    torch.manual_seed(42)
    model = make_lstm(n_layers)
    val_accs, elapsed = train_model(model, train_loader, val_loader)
    stack_results[n_layers] = {'val_accs': val_accs, 'time': elapsed}
    print(f'  Done. Final Val Acc: {val_accs[-1]:.4f}, Time: {elapsed:.1f}s')

fig, ax = plt.subplots(figsize=(10, 5))
for n_layers in [1, 2, 3]:
    ax.plot(epochs_range, stack_results[n_layers]['val_accs'],
            label=f'{n_layers}-Layer LSTM', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Effect of Stacking LSTM Layers on Validation Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

### How to Read This Chart: Effect of Stacking LSTM Layers

This line chart shows validation accuracy over 5 epochs for 1-layer, 2-layer, and 3-layer LSTMs.

- **The x-axis** is the training epoch.
- **The y-axis** is validation accuracy.
- **Each line** represents a different depth of LSTM stack.

**What to look for:**
- If the 2-layer LSTM outperforms the 1-layer, stacking is helping the model learn more complex patterns.
- If the 3-layer LSTM performs **worse** than the 2-layer, that's a sign of **overfitting** — more depth is hurting generalization on this dataset size.
- Convergence speed (how fast accuracy rises from epoch 1 to epoch 5) tells you whether adding layers helps or hurts early learning.

> **Key insight:** For the 20 Newsgroups dataset (relatively small), 2 layers is often the sweet spot. 3 layers can overfit without strong regularization or a larger dataset.

---

## Section 12 — Bahdanau Attention over LSTM

### The Problem with Final Hidden State Classification

All models so far use only the **last hidden state** for classification. But what if the most important word is at the beginning of the text? The LSTM must compress the entire sequence into a single vector — and early information can be diluted.

### Bahdanau Attention: A Better Approach

Instead of using just the final hidden state, attention computes a **weighted sum of ALL hidden states**:

```
score(h_t) = v^T · tanh(W · h_t)            ← How important is step t?
α_t = softmax(score(h_t)) over all t         ← Normalize to sum to 1
context = Σ α_t · h_t                        ← Weighted sum of all hidden states
output = classify(context)                   ← Use context for final prediction
```

The model **learns which words to pay attention to** for each classification decision. We can visualize these attention weights to understand model behavior.

In [ ]:
class AttentionLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, batch_first=True,
                            bidirectional=True,
                            dropout=DROPOUT if NUM_LAYERS > 1 else 0)
        self.drop = nn.Dropout(DROPOUT)
        # Additive (Bahdanau) attention
        self.attn_w = nn.Linear(HIDDEN_SIZE * 2, HIDDEN_SIZE)
        self.attn_v = nn.Linear(HIDDEN_SIZE, 1, bias=False)
        self.fc = nn.Linear(HIDDEN_SIZE * 2, NUM_CLASSES)

    def forward(self, x, return_attn=False):
        emb = self.drop(self.embed(x))          # (B, T, E)
        out, _ = self.lstm(emb)                 # (B, T, H*2)
        # Compute attention scores
        energy = torch.tanh(self.attn_w(out))   # (B, T, H)
        scores = self.attn_v(energy).squeeze(-1) # (B, T)
        attn_weights = torch.softmax(scores, dim=1)  # (B, T)
        # Weighted sum
        context = (attn_weights.unsqueeze(-1) * out).sum(dim=1)  # (B, H*2)
        logits = self.fc(self.drop(context))
        if return_attn:
            return logits, attn_weights
        return logits

print('Training Attention-BiLSTM...')
torch.manual_seed(42)
attn_model = AttentionLSTM()
attn_val_accs, attn_time = train_model(attn_model, train_loader, val_loader)
results['BiLSTM+Attention'] = {'val_accs': attn_val_accs, 'time': attn_time,
                                'final_acc': attn_val_accs[-1]}
print(f'Final Val Acc: {attn_val_accs[-1]:.4f}')

In [ ]:
# Visualize attention weights for 3 test examples
idx2word = {v: k for k, v in word2idx.items()}

attn_model.eval()
sample_X = torch.tensor(X_val_raw[:3], dtype=torch.long)
sample_y = y_val_raw[:3]

with torch.no_grad():
    logits, attn_weights = attn_model(sample_X, return_attn=True)
    preds = logits.argmax(dim=1).numpy()

label_names = {0: 'hockey', 1: 'sci.med'}
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for i in range(3):
    weights = attn_weights[i].numpy()  # (100,)
    tokens = [idx2word.get(int(sample_X[i][j].item()), '<UNK>') for j in range(MAX_LEN)]
    # Show only first 30 non-pad tokens for readability
    non_pad = [(t, w) for t, w in zip(tokens, weights) if t != '<PAD>']
    non_pad = non_pad[:30]
    tok_labels, tok_weights = zip(*non_pad) if non_pad else ([''], [0.0])

    ax = axes[i]
    ax.bar(range(len(tok_labels)), tok_weights, color='steelblue', edgecolor='white')
    ax.set_xticks(range(len(tok_labels)))
    ax.set_xticklabels(tok_labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Attention Weight')
    ax.set_title(f'Example {i+1} | True: {label_names[sample_y[i]]} | '
                 f'Predicted: {label_names[preds[i]]}')

plt.suptitle('Attention Weights Over Input Tokens (First 30 Non-Pad Tokens)', y=1.01)
plt.tight_layout()
plt.show()

### How to Read This Chart: Attention Weights Over Input Tokens

Each of the three subplots shows the attention weight distribution over the first 30 non-padding tokens of a test document.

- **The x-axis** shows the actual words in the document (first 30 non-pad tokens).
- **The y-axis** shows the attention weight assigned to each word — how much the model "looks at" that word when making its classification decision.
- **Taller bars** = the model considers this word more important for the classification.
- **Short bars** = the model largely ignores these words.

**What to look for:**
- For hockey documents, are words like "game", "team", "season", "player" getting high attention?
- For sci.med documents, are words like "patients", "treatment", "drug", "clinical" getting high attention?
- If attention is spread evenly, the model hasn't learned to focus — it's using all words equally.
- **Peaked attention** (one or two words much higher than the rest) suggests the model has identified the most discriminative words.

> **Why this matters:** Attention weights are one of the few ways we can interpret what a sequence model is "thinking about." High attention on domain-specific words is evidence the model has learned real linguistic patterns.

---

## Section 13 — Final Model Comparison Table

Here is a summary of all five models trained in this notebook:

| Model | Val Accuracy | Approx Params | Handles Long Sequences | Notes |
|-------|-------------|---------------|------------------------|-------|
| Vanilla RNN | ~75-80% | ~680K | Poor | Fast but forgets early tokens |
| LSTM | ~85-88% | ~1.05M | Good | Gated cell state solves vanishing gradient |
| GRU | ~84-87% | ~790K | Good | ~25% fewer params than LSTM, similar accuracy |
| BiLSTM | ~87-90% | ~2.1M | Good | Reads both directions; best single-direction model |
| BiLSTM + Attention | ~88-92% | ~2.2M | Excellent | Focuses on key tokens; best interpretability |

*(Actual numbers will vary based on random seed and training details)*

### Key Observations

1. **Every gated model (LSTM, GRU) outperforms Vanilla RNN** — gates are essential for NLP.
2. **Bidirectionality helps** — reading the sequence both ways gives the model full context at every position.
3. **Attention adds a small but consistent boost** — and makes the model interpretable.
4. **GRU is the efficiency winner** — close accuracy to LSTM with 25% fewer parameters and faster training.

---

## Summary and Decision Guide

### When to Use Which Sequence Model

**Use Vanilla RNN when:**
- The sequences are very short (< 10 steps)
- You need the absolute fastest training time
- You're building a proof of concept

**Use LSTM when:**
- Sequences are medium to long (20–500 tokens)
- You have long-range dependencies (the meaning of token 1 matters for predicting token 100)
- You have enough data to learn the extra parameters

**Use GRU when:**
- Same conditions as LSTM but you have limited compute
- You want faster training with nearly the same accuracy

**Use BiLSTM when:**
- Context from both sides matters (most NLP tasks)
- You're doing tagging, parsing, or classification (NOT generation)
- You can afford 2x the compute of a unidirectional LSTM

**Use BiLSTM + Attention when:**
- You need interpretability (attention weights tell you why)
- Sequences are long and the key information is sparse
- You're doing question answering or document classification

### Common Mistakes to Avoid

1. **Not using gradient clipping** — RNNs explode without it. Always use `clip_grad_norm_(..., 1.0)`.
2. **Processing pad tokens** — Use `pack_padded_sequence` for variable-length sequences.
3. **Using final hidden state for BiLSTM** — You must concatenate the last forward AND last backward hidden state.
4. **Forgetting dropout between stacked layers** — Without dropout, stacked RNNs overfit quickly.
5. **Using RNNs for very long sequences (> 512 tokens)** — Switch to Transformers. RNNs, even with LSTM, struggle beyond ~500 tokens.

### In 2024 and Beyond

For most NLP tasks, **Transformers (BERT, GPT, T5) have replaced RNNs** as the go-to architecture. However, sequence models remain relevant for:
- **Time series** (stock prices, sensor data, biomedical signals)
- **Low-resource environments** (edge devices with limited memory)
- **Streaming data** (where you process tokens one at a time in real time)
- **Understanding the foundations** of attention and transformers — everything in the next notebook builds on what you learned here.

---

*Next: Notebook 04 — Attention Mechanisms and Transformers: From Scratch to Fine-Tuning*